# Network Intrusion Detection
Payas Vaishnav · 23BCP430 · CSE PDEU · Semester 7

This executed companion notebook reads the real experiment outputs and demonstrates saved-model inference. Run `train.py` first to regenerate the benchmark. No training metrics are hard-coded here.

In [1]:
from pathlib import Path
import json
import pandas as pd
import joblib
from ids import ARTIFACTS, load_dataset, read_csv, predict
metrics = json.loads((ARTIFACTS / 'metrics.json').read_text())
print('Selected model:', metrics['best_model'])

Selected model: Histogram Gradient Boosting


## 1 Understand the inputs
A row describes a network flow. Feature columns contain traffic measurements; `Label` is the known class. Duplicating flows across partitions would make evaluation too easy.

In [2]:
x, y, removed = load_dataset()
print('Distinct rows:', len(x), 'Features:', len(x.columns), 'Duplicates removed:', removed)
print(y.value_counts().to_string())
print(x.iloc[:3, :5].to_string(index=False))

Distinct rows: 44051 Features: 77 Duplicates removed: 12610
Label
BENIGN          21688
DoS             16430
WebAttack        2143
BruteForce       1987
Bot              1441
PortScan          326
Infiltration       36
 Flow Duration  Total Fwd Packets  Total Backward Packets  Total Length of Fwd Packets  Total Length of Bwd Packets
             4                  2                       0                           37                            0
        142377                 46                      62                         1325                       105855
        118873                 23                      28                         1169                        45025


## 2 Inspect partition separation
The split uses seed 42 and stratification. Imputation and feature selection fit inside model pipelines. Three-fold training CV supports comparison; validation macro F1 chooses the final model.

In [3]:
partitions = pd.read_csv(ARTIFACTS / 'partitions.csv')
assert partitions.row_id.is_unique
print(partitions.partition.value_counts().to_string())

partition
train         26430
test           8811
validation     8810


## 3 Compare algorithms
The majority baseline demonstrates why overall accuracy alone is insufficient. Hyperparameter search uses only the training partition. The tuned tree CV result is a reused-search-fold diagnostic.

In [4]:
comparison = pd.read_csv(ARTIFACTS / 'comparison.csv')
print(comparison[['model','validation_accuracy','validation_macro_f1','cv_macro_f1_mean']].to_string(index=False))
print('Chosen tree settings:', metrics['tuning_best_params'])

                      model  validation_accuracy  validation_macro_f1  cv_macro_f1_mean
          Majority baseline             0.492395             0.094267          0.094258
              Decision Tree             0.993530             0.964125          0.907574
        Tuned Decision Tree             0.992736             0.954591          0.908137
              Random Forest             0.989444             0.927682          0.916592
                Extra Trees             0.983541             0.948084          0.895779
Histogram Gradient Boosting             0.993303             0.965283          0.929567
       Top 20 features Tree             0.984222             0.914316          0.894836
          Stacking ensemble             0.984109             0.850270          0.847810
Chosen tree settings: {'decisiontreeclassifier__min_samples_leaf': 1, 'decisiontreeclassifier__max_depth': None, 'decisiontreeclassifier__criterion': 'gini'}


## 4 Read final evaluation
Macro F1 weights every label equally. Per-class recall exposes missed attacks that accuracy may hide. The fixed test split has also been evaluated in the earlier prototype, so this remains an internal benchmark.

In [5]:
print('Test accuracy:', round(metrics['test_accuracy'],6))
print('Test macro F1:', round(metrics['test_macro_f1'],6))
print(pd.DataFrame({k:metrics['classification_report'][k] for k in metrics['labels']}).T.to_string())
print('Binary matrix [normal, attack]:', metrics['binary_confusion_matrix'])
print('95% macro F1 bootstrap interval:', metrics['macro_f1_ci95'])

Test accuracy: 0.996141
Test macro F1: 0.992333
              precision    recall  f1-score  support
BENIGN         0.997919  0.994929  0.996422   4338.0
Bot            0.966216  0.993056  0.979452    288.0
BruteForce     0.997481  0.994975  0.996226    398.0
DoS            0.996662  0.999391  0.998025   3286.0
Infiltration   1.000000  1.000000  1.000000      7.0
PortScan       1.000000  0.969231  0.984375     65.0
WebAttack      0.992991  0.990676  0.991832    429.0
Binary matrix [normal, attack]: [[4316, 22], [9, 4464]]
95% macro F1 bootstrap interval: [0.9881799400681692, 0.9954922939284571]


## 5 Predict new flow records
Use the supplied held-out demo flows. An uploaded file needs the same columns and measurement units. Confidence scores are uncalibrated model scores.

In [6]:
bundle = joblib.load(ARTIFACTS / 'model.joblib')
flows = read_csv(ARTIFACTS / 'demo_flows.csv')
result = predict(bundle, flows)
print(result[['Predicted label','Traffic status','Model confidence']].to_string(index=False))

Predicted label Traffic status  Model confidence
         BENIGN         Normal          0.999767
         BENIGN         Normal          0.999804
            DoS         Attack          0.999691
         BENIGN         Normal          0.999822
         BENIGN         Normal          0.999811
         BENIGN         Normal          0.997579
            DoS         Attack          0.999746
         BENIGN         Normal          0.999826
         BENIGN         Normal          0.999810
         BENIGN         Normal          0.999798
         BENIGN         Normal          0.999205
            DoS         Attack          0.999755
         BENIGN         Normal          0.998121
            DoS         Attack          0.968227
         BENIGN         Normal          0.999590
            DoS         Attack          0.999618
         BENIGN         Normal          0.992777
            DoS         Attack          0.998567
         BENIGN         Normal          0.610544
            DoS     

## 6 Examine mistakes
Retain errors for discussion. A missed attack and a false alarm have different practical costs. A perfect score from only a few rare examples is weak evidence.

In [7]:
errors = pd.read_csv(ARTIFACTS / 'error_examples.csv')
print(errors[['actual','predicted']].value_counts().to_string())
print('Total multiclass mistakes:', len(errors))

actual      predicted 
BENIGN      Bot           10
            DoS            8
            WebAttack      3
Bot         BENIGN         2
WebAttack   BENIGN         2
            DoS            2
DoS         BENIGN         2
BruteForce  BENIGN         2
PortScan    BENIGN         1
            DoS            1
BENIGN      BruteForce     1
Total multiclass mistakes: 34


## 7 Reproduce and extend
Run `python train.py` using the project virtual environment, then `python verify.py`. The report and presentation sources are in `tools/`. A stronger extension requires the full dataset with day or session metadata for an independent chronological evaluation. The current sample does not provide those grouping fields.

References: Western OC2 Lab IDS-ML repository, CICIDS2017 dataset documentation and scikit-learn documentation, linked in the README.